In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    models,
)
from sentence_transformers.training_args import BatchSamplers
from transformers import BitsAndBytesConfig


In [ ]:
# --- 1. Configure Quantization (The "Q" in QLoRA) ---
# This tells the model to load in 4-bit NF4 format to save massive memory
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


# --- 2. Load Quantized Llama 3.2 ---
model_name = "meta-llama/Llama-3.2-1B"

# 1. Load the Base Transformer
word_embedding_model = models.Transformer(
    model_name,
    max_seq_length=2048,  # Adjust as needed
    model_args={
        "quantization_config": bnb_config,
        "trust_remote_code": True,
        # "attn_implementation": "flash_attention_2"
    },
)

# 2. Add the Pooling Layer (Explicitly choosing 'last_token')
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode="lasttoken",
)

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# --- 3. Attach LoRA Adapter ---
# You cannot train a 4-bit model directly. You must attach an adapter.
peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model.model = get_peft_model(model.model, peft_config)

# --- 4. Training Setup (Same as before) ---
# Since Llama 3.2 3B is so small, you can use a HUGE batch size on an H100.
# Try batch_size=256 or even 512 for massive in-batch negative signal.

args = SentenceTransformerTrainingArguments(
    output_dir="llama-3.2-3b-embedding-quantized",
    num_train_epochs=1,
    per_device_train_batch_size=128,  # Crank this up!
    learning_rate=2e-4,  # QLoRA often needs slightly higher LR
    warmup_ratio=0.1,
    fp16=False,
    bf16=True,  # Always use bf16 on H100
    batch_sampler=BatchSamplers.NO_DUPLICATES,
)

# Define your data (Anchor/Positive pairs)
data = [
    {
        "anchor": "How to optimize Python code?",
        "positive": "Use vectorization with NumPy...",
    },
    {"anchor": "Capital of France", "positive": "Paris is the capital..."},
    # ... your dataset
]
train_dataset = Dataset.from_list(data)
train_loss = losses.MultipleNegativesRankingLoss(model)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
)

trainer.train()

# --- 5. Saving ---
# This saves ONLY the adapter (MBs, not GBs)
model.save_pretrained("llama-3.2-3b-embed-final")

In [ ]:
# 1. Create a dummy dataset (Anchor = Product Name, Positive = Description)
data = [
    {
        "anchor": "AmazonBasics USB-C to USB-A 2.0 Fast Charging Cable",
        "positive": "Connect USB Type-C enabled devices to standard USB Type-A enabled devices with this cable.",
    },
    {
        "anchor": "Logitech MX Master 3S Wireless Mouse",
        "positive": "Performance wireless mouse with 8K DPI tracking and quiet clicks.",
    },
    {
        "anchor": "Sony WH-1000XM5 Wireless Noise Canceling Headphones",
        "positive": "Industry-leading noise canceling with two processors controlling 8 microphones.",
    },
    {
        "anchor": "Samsung T7 Shield Portable SSD 2TB",
        "positive": "Rugged, fast, and compact external storage with IP65 rating for dust and water resistance.",
    },
    {
        "anchor": "Kindle Paperwhite (16 GB)",
        "positive": "Now with a 6.8” display and adjustable warm light, up to 10 weeks of battery life.",
    },
    {
        "anchor": "Anker 737 Power Bank (PowerCore 24K)",
        "positive": "Ultra-powerful two-way charging with 140W max output and smart digital display.",
    },
]

dataset = Dataset.from_list(data)
print("Dataset created:", dataset)

In [ ]:
# 2. Prepare the model for QLoRA training
# We access the underlying transformer via model[0].auto_model

# Ensure the tokenizer has a pad token (Llama usually doesn't by default)
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model[0].auto_model = get_peft_model(model[0].auto_model, peft_config)
model[0].auto_model.print_trainable_parameters()

In [ ]:
# 3. Unsupervised Training (SimCSE / Contrastive Tension)
# Now that LoRA is applied, we can run unsupervised training.

# ContrastiveTensionLossInBatchNegatives expects two identical inputs to compute the loss.
# We create a dataset where both columns contain the same sentences.
unsupervised_corpus = [row["anchor"] for row in data]
unsupervised_dataset = Dataset.from_dict(
    {"sentence1": unsupervised_corpus, "sentence2": unsupervised_corpus}
)

# Initialize the specific loss for unsupervised training
# This loss effectively performs SimCSE by using dropout as data augmentation
train_loss_ct = losses.ContrastiveTensionLossInBatchNegatives(model=model)

args_ct = SentenceTransformerTrainingArguments(
    output_dir="llama-3.2-1b-ct-loss",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=1,
    report_to="none",
)

trainer_ct = SentenceTransformerTrainer(
    model=model,
    train_dataset=unsupervised_dataset,
    loss=train_loss_ct,
    args=args_ct,
)

trainer_ct.train()

In [ ]:
# 3. Set up Training

# MultipleNegativesRankingLoss is ideal for (anchor, positive) pairs
train_loss = losses.MultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir="llama-3.2-1b-lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=dataset,
    loss=train_loss,
    args=args,
)

trainer.train()

In [ ]:
# 3. Unsupervised Training (SimCSE / Contrastive Tension)
# Now that LoRA is applied, we can run unsupervised training.

# Create datasets where both columns contain the same sentences for Contrastive Tension
unsupervised_train_corpus = df_train["anchor"].tolist()
unsupervised_val_corpus = df_val["anchor"].tolist()

unsupervised_train_dataset = Dataset.from_dict(
    {"sentence1": unsupervised_train_corpus, "sentence2": unsupervised_train_corpus}
)

unsupervised_eval_dataset = Dataset.from_dict(
    {"sentence1": unsupervised_val_corpus, "sentence2": unsupervised_val_corpus}
)

# Initialize the specific loss for unsupervised training
train_loss_ct = losses.ContrastiveTensionLossInBatchNegatives(model=model)

args_ct = SentenceTransformerTrainingArguments(
    output_dir="llama-3.2-1b-ct-loss",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,  # Evaluate every 500 steps
    save_strategy="steps",
    save_steps=1000,
    report_to="wandb",
    run_name="llama-3.2-1b-unsupervised",
)

trainer_ct = SentenceTransformerTrainer(
    model=model,
    train_dataset=unsupervised_train_dataset,
    eval_dataset=unsupervised_eval_dataset,
    loss=train_loss_ct,
    args=args_ct,
)

trainer_ct.train()

# IMPORTANT: Close the run so the next training starts fresh
wandb.finish()